# Audit California ticket schedule amounts
Compare 49 dated 2026 references across six counties, reproduce one reviewed calculation and inspect the cited pages. These examples assume an ordinary adult first offense. The court determines your case balance, due date and eligibility. Data curated by California Licensing Education under CC BY 4.0. [Data and scope](https://github.com/Nathan-Roll1/california-traffic-ticket-and-tvs-fees-2026/tree/v2026-11-17.3) · [School tools](https://traffic.californialicensingeducation.com/tools/) · [Cost guide](https://traffic.californialicensingeducation.com/traffic-school-cost/)

In [ ]:
import hashlib, json, subprocess, sys, tempfile, urllib.request
from pathlib import Path
release = "https://github.com/Nathan-Roll1/california-ticket-decoder/releases/download/v0.1.0/"
tmp = tempfile.TemporaryDirectory()
target = Path(tmp.name)
wheel = target / "california_ticket_decoder-0.1.0-py3-none-any.whl"
wheel.write_bytes(urllib.request.urlopen(release + wheel.name, timeout=30).read())
assert hashlib.sha256(wheel.read_bytes()).hexdigest() == "001203f265414461bfc05c831b1900425e5cc1f6cfe4c8522e028fe74569643d"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-deps", "--target", str(target / "lib"), str(wheel)], check=True)
sys.path.insert(0, str(target / "lib"))
from california_ticket_decoder import load
rows = load("ticket-references-live")
sources = load("sources")
assert len(rows) == len({r["id"] for r in rows}) == 49
for county in sorted({r["county"] for r in rows}):
    print(county, sum(r["county"] == county for r in rows), "reviewed rows")

In [ ]:
# Change these three values to audit a different row. Use None when there is no speed band.
county, subsection, speed_band = "Santa Barbara", "22450(a)", None
row = next((r for r in rows if (r["county"], r["code"], r["speedBand"]) == (county, subsection, speed_band)), None)
if row is None:
    print("No reviewed match. Check the county, exact subsection and speed band.")
else:
    print("Kind:", row["referenceKind"], "| Reviewed:", row["reviewedAt"])
    for name, cents in row["componentsCents"].items():
        print(f"{name:26} ${cents / 100:,.2f}")
    print(f"Scheduled total: ${row['scheduledTotalCents'] / 100:,.2f}")
    refs = row.get("calculation", {}).get("sourceReferences") or [{"sourceId": row["sourceId"], "pdfPages": [row["pdfPage"]]}]
    for ref in refs:
        print(sources[ref["sourceId"]]["url"], "PDF pages", ref["pdfPages"])

In [ ]:
# Reconcile each row in cents. Court traffic-school fees and tuition are separate.
mismatches = [r["id"] for r in rows if sum(r["componentsCents"].values()) != r["scheduledTotalCents"]]
assert not mismatches, mismatches
print("All 49 component sums reconcile to the scheduled total.")